# 0201_ingest_csv_file
Lands one CSV file (the mapping workbook) into Bronze as delivered: every column as text,
typing happens in Silver. Same audit columns and the same snapshot history as the parquet loader.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## Loader

In [ ]:
def ingest_csv_file(config, batch_id):
    source_folder = CONNECTIONS[config["source_connection_name"]]
    source_file = source_folder / config["source_object"]
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", "UTF-8")
        .csv(source_file.as_posix())
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(source_file.name))
        .withColumn("_batch_id", F.lit(batch_id))
    )
    if config["load_type"] == "SNAPSHOT":
        return write_snapshot(df, "bronze", config["destination_schema"], config["destination_table"])
    return write_table(df, "bronze", config["destination_schema"], config["destination_table"])


LOADERS["0201_ingest_csv_file"] = ingest_csv_file

## Run on its own

In [ ]:
# Parameters: set a control_id and run all to load one flow on its own (handy for testing).
# Left at 0, this notebook only defines its loader for the orchestrator.
control_id = 0
batch_id = ""

In [ ]:
if control_id:
    print(execute_flow(control_id, batch_id or new_batch_id(), ingest_csv_file, triggered_by="manual"))